In [1]:
import pandas as pd
import glob
import os

Üretim Verileri

In [2]:
# Tüm csv dosyalarını al
files = glob.glob("C:/Users/kerem/Desktop/elektrikveri/Üretim/*.csv")

df_list = []

for file in files:
    df = pd.read_csv(
        file,
        sep=";",              # Eğer csv ; ile ayrılmışsa
        decimal=",",          # 33308,67 formatı için
        dayfirst=True,         # 1.10.2021 formatı için
        encoding="utf-8-sig"
    )
    df_list.append(df)

# Hepsini alt alta birleştir
üretim = pd.concat(df_list, ignore_index=True)

print(üretim.head())

       Tarih   Saat    Toplam
0  1.01.2021  00:00  29488.11
1  1.01.2021  01:00  28065.76
2  1.01.2021  02:00  26527.08
3  1.01.2021  03:00  25327.19
4  1.01.2021  04:00  24719.72


In [3]:
üretim.to_csv("tüm_üretim.csv", index=False)

In [4]:
üretim.rename(columns={"Toplam": "Üretim Miktarı (MWh)"}, inplace=True)

In [5]:
üretim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Tarih                 43824 non-null  object 
 1   Saat                  43824 non-null  object 
 2   Üretim Miktarı (MWh)  43824 non-null  float64
dtypes: float64(1), object(2)
memory usage: 1.0+ MB


In [6]:
üretim["Datetime"] = pd.to_datetime(
    üretim["Tarih"] + " " + üretim["Saat"],
    dayfirst=True
)

üretim = üretim[["Datetime", "Üretim Miktarı (MWh)"]]

Tüketim Verileri

In [7]:
# Tüm csv dosyalarını al
files = glob.glob("C:/Users/kerem/Desktop/elektrikveri/Tüketim/*.csv")

df_list = []

for file in files:
    df = pd.read_csv(
        file,
        sep=";",              # Eğer csv ; ile ayrılmışsa
        decimal=",",          # 33308,67 formatı için
        dayfirst=True,
        encoding="utf-8-sig" # 1.10.2021 formatı için
    )
    df_list.append(df)

# Hepsini alt alta birleştir
tüketim = pd.concat(df_list, ignore_index=True)

print(tüketim.head())

        Tarih   Saat Tüketim Miktarı(MWh)
0  01.01.2021  00:00            29.489,46
1  01.01.2021  01:00            28.067,11
2  01.01.2021  02:00            26.527,08
3  01.01.2021  03:00            25.327,19
4  01.01.2021  04:00            24.719,72


In [8]:
tüketim.to_csv("tüm_tüketim.csv", index=False)

In [9]:
tüketim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Tarih                 43824 non-null  object
 1   Saat                  43824 non-null  object
 2   Tüketim Miktarı(MWh)  43824 non-null  object
dtypes: object(3)
memory usage: 1.0+ MB


In [10]:
tüketim["Datetime"] = pd.to_datetime(
    tüketim["Tarih"] + " " + tüketim["Saat"],
    dayfirst=True
)

# 29.489,46 → 29489.46 dönüştürme
tüketim["Tüketim Miktarı(MWh)"] = (
    tüketim["Tüketim Miktarı(MWh)"]
    .str.replace(".", "", regex=False)   # binlik noktayı sil
    .str.replace(",", ".", regex=False)  # virgülü noktaya çevir
    .astype(float)
)

tüketim = tüketim[["Datetime", "Tüketim Miktarı(MWh)"]]

Fiyat Verileri

In [11]:
# Tüm csv dosyalarını al
files = glob.glob("C:/Users/kerem/Desktop/elektrikveri/Fiyat/*.csv")

df_list = []

for file in files:
    df = pd.read_csv(
        file,
        sep=";",              # Eğer csv ; ile ayrılmışsa
        decimal=",",          # 33308,67 formatı için
        dayfirst=True,
        encoding="utf-8-sig" # 1.10.2021 formatı için
    )
    df_list.append(df)

# Hepsini alt alta birleştir
fiyat = pd.concat(df_list, ignore_index=True)

print(fiyat.head())

              Tarih   Saat AOF (TL/MWh)
0  01.01.2021 00:00  00:00       259,39
1  01.01.2021 01:00  01:00       234,13
2  01.01.2021 02:00  02:00       215,25
3  01.01.2021 03:00  03:00       219,24
4  01.01.2021 04:00  04:00       209,81


In [12]:
fiyat.to_csv("tüm_fiyat.csv", index=False)

In [13]:
fiyat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Tarih         43824 non-null  object
 1   Saat          43824 non-null  object
 2   AOF (TL/MWh)  43807 non-null  object
dtypes: object(3)
memory usage: 1.0+ MB


In [14]:
fiyat["Datetime"] = pd.to_datetime(
    fiyat["Tarih"],
    dayfirst=True,
    format="%d.%m.%Y %H:%M"   # kesin format → hızlı ve hatasız
)

fiyat["AOF (TL/MWh)"] = (
    fiyat["AOF (TL/MWh)"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

fiyat = fiyat[["Datetime", "AOF (TL/MWh)"]].rename(
    columns={"AOF (TL/MWh)": "Ortalama Fiyat (TL/MWh)"}
)

print(f"Fiyat datetime örnek: {fiyat['Datetime'].iloc[0]}")  # 2021-01-01 00:00:00 olmalı
print(f"Fiyat satır sayısı  : {len(fiyat)}")

Fiyat datetime örnek: 2021-01-01 00:00:00
Fiyat satır sayısı  : 43824


Tüm Veriler

In [15]:
df_final = (
    üretim
    .merge(tüketim, on="Datetime", how="outer")
    .merge(fiyat,   on="Datetime", how="outer")
    .sort_values("Datetime")
    .reset_index(drop=True)
)

# ── Bütünlük kontrolü ────────────────────────────────────────────────────────
BEKLENEN_SATIR = 43824
print("=" * 50)
print("BİRLEŞTİRME DOĞRULAMA RAPORU")
print("=" * 50)
print(f"Beklenen satır : {BEKLENEN_SATIR}")
print(f"Gerçek satır   : {len(df_final)}")

if len(df_final) != BEKLENEN_SATIR:
    eksik = BEKLENEN_SATIR - len(df_final)
    print(f"{eksik} SATIR EKSİK — Hangi saatler boş?")
    # Tam tarih dizisi oluştur, hangisi eksik bul
    tam_index = pd.date_range(
        df_final["Datetime"].min(),
        df_final["Datetime"].max(),
        freq="h"
    )
    eksik_saatler = tam_index.difference(pd.DatetimeIndex(df_final["Datetime"]))
    print(f"Eksik saatler:\n{eksik_saatler}")
else:
    print(" Satır sayısı doğru")

print(f"\nNaN sayıları (outer join sonrası):")
print(df_final.isnull().sum())

# ── Tarih aralığı kontrolü ───────────────────────────────────────────────────
print(f"\nTarih aralığı : {df_final['Datetime'].min()} → {df_final['Datetime'].max()}")
print(f"İlk 5 satır:\n{df_final.head()}")

# ── Kaydet ───────────────────────────────────────────────────────────────────
df_final.to_csv("birleşik_veri.csv", index=False)
print("\n birleşik_veri.csv kaydedildi")

BİRLEŞTİRME DOĞRULAMA RAPORU
Beklenen satır : 43824
Gerçek satır   : 43824
✓ Satır sayısı doğru

NaN sayıları (outer join sonrası):
Datetime                    0
Üretim Miktarı (MWh)        0
Tüketim Miktarı(MWh)        0
Ortalama Fiyat (TL/MWh)    17
dtype: int64

Tarih aralığı : 2021-01-01 00:00:00 → 2025-12-31 23:00:00
İlk 5 satır:
             Datetime  Üretim Miktarı (MWh)  Tüketim Miktarı(MWh)  \
0 2021-01-01 00:00:00              29488.11              29489.46   
1 2021-01-01 01:00:00              28065.76              28067.11   
2 2021-01-01 02:00:00              26527.08              26527.08   
3 2021-01-01 03:00:00              25327.19              25327.19   
4 2021-01-01 04:00:00              24719.72              24719.72   

   Ortalama Fiyat (TL/MWh)  
0                   259.39  
1                   234.13  
2                   215.25  
3                   219.24  
4                   209.81  

✓ birleşik_veri.csv kaydedildi
